# CN7 가스7개 모델 비라벨 적용
전체605 라벨 학습＋기존 검증 기준 중앙값 고정. 원래OOF F1 .714 모델과 구분한다.
공식S14 비라벨 원시센서 연결, 알려진606샷 제외, 과거5 이력없는샷 보류. 정답없으므로 경보비율만 계산.


In [1]:
from pathlib import Path
import json,hashlib,warnings
import numpy as np,pandas as pd,nbformat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier,RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore",category=RuntimeWarning)
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
S=OUT/"cn7_reference_snapshot"
T=pd.read_csv(S/"cn7_163_gas_supervised_input_raw_shots.csv",parse_dates=["TimeStamp"]).sort_values(["run_id","TimeStamp"]).reset_index(drop=True)
assert len(T)==606 and T.gas.sum()==13
eligible=(T.gas|T.any_fail.eq(0)).to_numpy()
train=np.flatnonzero(eligible);normal=np.flatnonzero(T.any_fail.eq(0))
y=T.gas.to_numpy(int)
rawcols=[c for c in T if c.endswith("_raw")]
fields=[c[:-4] for c in rawcols]
official=pd.read_csv(ROOT/"data/moldset_unlabeled_cn7_with_timestamp.csv",parse_dates=["TimeStamp"])
raw=pd.read_csv(ROOT/"data/original_split/moldset_unlabeled_cn7.csv",parse_dates=["TimeStamp"])
raw=raw[raw.EQUIP_CD.eq("S14")].copy()
assert len(raw)==len(official)==35239
group=raw.groupby("TimeStamp",sort=True)
conflict=(group[fields].nunique()>1).any(axis=1)
U=group[fields].mean().rename(columns={c:c+"_raw" for c in fields}).reset_index()
U["row_count"]=group.size().reindex(U.TimeStamp).to_numpy()
U["sensor_conflict"]=conflict.reindex(U.TimeStamp).to_numpy()
assert U.TimeStamp.is_unique and len(U)==28744
known=pd.read_csv(OUT/"original_labeled_shot_features.csv",parse_dates=["TimeStamp"])
knownkeys=set(known.loc[known["product"].eq("CN7"),"TimeStamp"])
U["known_label"]=U.TimeStamp.isin(knownkeys)
assert U.known_label.sum()==606
officialkeys=set(official.TimeStamp)
assert set(U.TimeStamp)==officialkeys
# Physical run for HISTORY only. Reset on >30min gap or an intervening RG3 timestamp.
rg=pd.read_csv(ROOT/"data/original_split/moldset_unlabeled_rg3.csv",usecols=["TimeStamp","EQUIP_CD"],parse_dates=["TimeStamp"])
rgtime=np.sort(rg.loc[rg.EQUIP_CD.eq("S14"),"TimeStamp"].drop_duplicates().to_numpy())
times=U.TimeStamp.to_numpy()
between=np.zeros(len(U),bool)
for i in range(1,len(U)):
 j=np.searchsorted(rgtime,times[i-1],side="right")
 between[i]=j<len(rgtime) and rgtime[j]<times[i]
reset=U.TimeStamp.diff().dt.total_seconds().gt(1800).fillna(True).to_numpy()|between
# Conflicting LH/RH sensor records must not enter subsequent past5 history.
reset=reset|U.sensor_conflict.to_numpy()|U.sensor_conflict.shift(1,fill_value=False).to_numpy()
reset[0]=True
U["history_run"]=np.cumsum(reset)
# Training-only normalization modes. This is operating-condition matching, NOT true Run labels.
control=["Barrel_Temperature_5_raw","Hopper_Temperature_raw","Mold_Temperature_4_raw"]
imp=SimpleImputer().fit(T.loc[normal,control])
sc=RobustScaler().fit(imp.transform(T.loc[normal,control]))
cn=sc.transform(imp.transform(T.loc[normal,control]))
cu=sc.transform(imp.transform(U[control]))
knn=NearestNeighbors(n_neighbors=1).fit(cn)
distance,nn=knn.kneighbors(cu)
U["reference_mode"]=T.run_id.iloc[normal[nn[:,0]]].to_numpy()
# Support flag uses leave-self-out distances between labeled normals, no unlabeled quality assumption.
nd,_=NearestNeighbors(n_neighbors=2).fit(cn).kneighbors(cn)
support_cut=float(np.quantile(nd[:,1],.99,method="higher"))
U["in_reference_support"]=distance[:,0]<=support_cut+1e-12
U["reference_distance"]=distance[:,0]
combined=pd.concat([T[rawcols],U[rawcols]],ignore_index=True)
A=combined.to_numpy(float);q=np.arange(len(T),len(combined))
mode=np.r_[T.run_id.to_numpy(),U.reference_mode.to_numpy()]
def clean(a):
 return np.where(np.isfinite(a),a,np.nan)
def standard(a):
 ans=np.empty_like(a,dtype=float)
 for m in T.run_id.unique():
  ni=normal[T.run_id.iloc[normal].eq(m).to_numpy()]
  ii=np.flatnonzero(mode==m)
  imp=SimpleImputer(keep_empty_features=True).fit(clean(a[ni]));v=imp.transform(clean(a))
  floor=[]
  for k in range(v.shape[1]):
   dif=np.diff(np.unique(np.round(v[ni,k],4)));nz=dif[dif>1e-8]
   floor.append(max(float(np.min(nz)) if len(nz) else .01,1e-4))
  scale=np.maximum(v[ni].std(axis=0),floor)
  ans[ii]=(v[ii]-np.median(v[ni],axis=0))/scale
 return np.clip(ans,-20,20)
def relation_scores():
 relations=[("Max_Injection_Pressure",["Max_Injection_Speed","Filling_Time"]),
 ("Plasticizing_Time",["Max_Screw_RPM","Average_Back_Pressure"]),
 ("Cushion_Position",["Plasticizing_Position","Max_Injection_Pressure"]),
 ("Mold_Temperature_4",["Mold_Temperature_3","Barrel_Temperature_5"])]
 result=[]
 for target,preds in relations:
  cols=[v+"_raw" for v in preds+[target]]
  v=combined[cols].to_numpy(float)
  imp=SimpleImputer(keep_empty_features=True).fit(v[normal]);v=imp.transform(v)
  sc=RobustScaler().fit(v[normal,:-1]);reg=Ridge(alpha=10).fit(sc.transform(v[normal,:-1]),v[normal,-1])
  err=v[:,-1]-reg.predict(sc.transform(v[:,:-1]))
  scale=max(float(err[normal].std()),float(v[normal,-1].std())*.1,1e-6)
  result.append(err/scale)
 return np.column_stack(result)
R=relation_scores()
# Keep sparse label history to match the original training definition.
# Query history is chronological raw unlabeled history; not filled with quality predictions.
trhist=pd.DataFrame(np.abs(np.clip(R[:len(T)],-20,20))).groupby(T.run_id.to_numpy(),group_keys=False).apply(lambda a:a.shift(1).rolling(5,min_periods=3).mean()).to_numpy()
uhist=pd.DataFrame(np.abs(np.clip(R[len(T):],-20,20))).groupby(U.history_run.to_numpy(),group_keys=False).apply(lambda a:a.shift(1).rolling(5,min_periods=3).mean()).to_numpy()
H=np.vstack([trhist,uhist])
U["history_ready"]=np.isfinite(uhist).all(axis=1)
cuts={}
def frozen_cut(name,path,model,rule):
 a=pd.read_csv(path)
 a=a[(a.model==model)&(a.rule==rule)]
 assert len(a)==606
 values=a.threshold.drop_duplicates().to_numpy()
 cut=float(np.median(a.groupby("heldout_block").threshold.first())) if "heldout_block" in a else float(np.median(values))
 cuts[name]={"source":str(path),"model":model,"rule":rule,"frozen_cut":cut,"aggregation":"median of20 heldout-block thresholds, not fitted to unlabeled"}
 return cut
EXT=Path(r"C:/Users/hayeon/.codex/worktrees/b365/KAMP-IMM/reports/tables")
cut1=frozen_cut("relation_ET",EXT/"cn7_163_gas_supervised_predictions.csv","residual4_ET","inner_normal99")
cut2=frozen_cut("joint_LR",EXT/"cn7_164_gas_supervised_predictions.csv","joint_intensity4_LR","inner_normal99")
cut3=frozen_cut("load_LR",EXT/"cn7_164_gas_supervised_predictions.csv","load6_abs6_LR","inner_normal97")
cut4=frozen_cut("persistent_ET",OUT/"cn7_170_persistence_predictions.csv","persistent_abs4_ET","normal95")
cut5=frozen_cut("transfer_LR",OUT/"cn7_173_transfer_predictions.csv","mechanism_joint8_LR","inner_normal0.99")
cut6=frozen_cut("local_flow_RF",OUT/"cn7_175_magnitude_predictions.csv","local_flow_absolute6_RF","normal99")
cut7=frozen_cut("abs_flow_LR",OUT/"cn7_175_magnitude_predictions.csv","flow_absolute6_LR","normal99")
models={}
def fit_score(name,x,kind):
 imp=SimpleImputer(keep_empty_features=True).fit(clean(x[normal]))
 v=imp.transform(clean(x))
 sc=RobustScaler().fit(v[normal]);v=np.clip(sc.transform(v),-20,20)
 if kind=="old_ET":m=ExtraTreesClassifier(n_estimators=50,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)
 elif kind=="new_ET":m=ExtraTreesClassifier(n_estimators=40,max_depth=3,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)
 elif kind=="RF":m=RandomForestClassifier(n_estimators=40,max_depth=3,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)
 else:m=LogisticRegression(C=.1,class_weight="balanced",max_iter=3000)
 m.fit(v[train],y[train]);models[name]=(imp,sc,m)
 return m.predict_proba(v[q])[:,1]
score1=fit_score("relation_ET",R,"old_ET")
score4=fit_score("persistent_ET",H,"new_ET")
def col(c):return combined[c+"_raw"].to_numpy(float)
rpm=col("Max_Screw_RPM");bp=col("Average_Back_Pressure");pt=col("Plasticizing_Time")
p=col("Max_Injection_Pressure");sw=col("Max_Switch_Over_Pressure");speed=col("Max_Injection_Speed")
fill=col("Filling_Time");meter=col("Plasticizing_Position");cushion=col("Cushion_Position")
ratio=lambda a,b:np.divide(a,b,out=np.full_like(a,np.nan),where=np.abs(b)>1e-8)
load=np.column_stack([rpm,bp,ratio(bp,rpm),pt*rpm,col("Mold_Temperature_4")-col("Mold_Temperature_3"),col("Hopper_Temperature")])
transition=np.column_stack([p-sw,ratio(sw,p),ratio(p,speed),fill*speed,meter-cushion,pt])
oldcontrols=np.column_stack([col("Barrel_Temperature_5"),col("Hopper_Temperature"),col("Mold_Temperature_3"),speed])
def conditional(x,ctrl):
 imp=SimpleImputer(keep_empty_features=True).fit(clean(ctrl[normal]));cx=imp.transform(clean(ctrl))
 sc=RobustScaler().fit(cx[normal]);cx=sc.transform(cx)
 imp=SimpleImputer(keep_empty_features=True).fit(clean(x[normal]));xx=imp.transform(clean(x))
 floor=np.maximum(xx[normal].std(axis=0)*.1,1e-6);out=np.empty_like(xx)
 for m in T.run_id.unique():
  ni=normal[T.run_id.iloc[normal].eq(m).to_numpy()]
  ids=np.flatnonzero(mode==m)
  neighbors=NearestNeighbors(n_neighbors=min(31,len(ni))).fit(cx[ni]).kneighbors(cx[ids],return_distance=False)
  for i,ns in zip(ids,neighbors):
   peers=ni[ns];peers=peers[peers!=i][:30]
   out[i]=(xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor)
 return out
lz=np.abs(conditional(load,oldcontrols))
tz=np.abs(conditional(transition,oldcontrols))
z=np.clip(np.column_stack([lz,tz]),0,20)
joint=np.column_stack([z.max(axis=1),np.sort(z,axis=1)[:,-2],z.mean(axis=1),(z>2).sum(axis=1)])
score2=fit_score("joint_LR",joint,"LR")
score3=fit_score("load_LR",lz,"LR")
BASIC=["Mold_Temperature_3","Mold_Temperature_4","Max_Injection_Speed","Filling_Time","Max_Injection_Pressure","Max_Switch_Over_Pressure","Plasticizing_Position","Plasticizing_Time","Cushion_Position","Average_Back_Pressure","Max_Screw_RPM"]
Z=standard(np.column_stack([col(c) for c in BASIC]))
m3,m4,sp,fi,pr,switch,me,pl,cu,bpa,rp=Z.T
flow=np.column_stack([m3,sp,fi,pr,switch,pr-sp])
score7=fit_score("abs_flow_LR",np.abs(flow),"LR")
# New175 conditional features use a sensor-resolution floor, unlike164.
newcontrols=np.column_stack([col("Barrel_Temperature_5"),col("Hopper_Temperature"),col("Mold_Temperature_4")])
newraw=np.column_stack([col("Mold_Temperature_3"),speed,fill,p,p-sw,meter])
def conditional_new(x,ctrl):
 imp=SimpleImputer(keep_empty_features=True).fit(ctrl[normal]);cx=imp.transform(ctrl)
 sc=RobustScaler().fit(cx[normal]);cx=sc.transform(cx)
 imp=SimpleImputer(keep_empty_features=True).fit(x[normal]);xx=imp.transform(x)
 floor=[]
 for k in range(xx.shape[1]):
  dif=np.diff(np.unique(np.round(xx[normal,k],4)));nz=dif[dif>1e-8]
  step=max(float(np.min(nz)) if len(nz) else .01,1e-4)
  floor.append(max(step,float(xx[normal,k].std())*.1))
 out=np.empty_like(xx)
 for m in T.run_id.unique():
  ni=normal[T.run_id.iloc[normal].eq(m).to_numpy()];ids=np.flatnonzero(mode==m)
  nn=NearestNeighbors(n_neighbors=min(31,len(ni))).fit(cx[ni]).kneighbors(cx[ids],return_distance=False)
  for i,n in zip(ids,nn):
   peers=ni[n];peers=peers[peers!=i][:30]
   out[i]=(xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor)
 return np.clip(out,-20,20)
score6=fit_score("local_flow_RF",np.abs(conditional_new(newraw,newcontrols)),"RF")
def embedding(z):
 m3,m4,sp,fi,pr,sw,me,pl,cu,bp,rpm=z.T
 return np.column_stack([np.minimum(-m3,sp),np.minimum(-fi,sp),np.minimum(-pr,sp),np.minimum(-m3,-fi),np.minimum(me,pl),np.minimum(cu,pl),np.minimum(bp,-rpm),np.abs(m4-m3)])
D=known[known["product"].eq("RG3")].copy()
D["gas"]=D.Reason.fillna("").str.contains("가스")
D=D[D.gas|D.PassOrFail.eq(0)].reset_index(drop=True)
assert len(D)==583 and D.gas.sum()==17
dz=np.empty((len(D),len(BASIC)))
for m in D.comparison_block.unique():
 ids=np.flatnonzero(D.comparison_block.eq(m));rr=D[BASIC].iloc[ids].to_numpy(float)
 ni=D.gas.iloc[ids].eq(0).to_numpy()
 imp=SimpleImputer(keep_empty_features=True).fit(rr[ni]);rr=imp.transform(rr)
 floor=[]
 for k in range(rr.shape[1]):
  dif=np.diff(np.unique(np.round(rr[ni,k],4)));nz=dif[dif>1e-8]
  floor.append(max(float(nz.min()) if len(nz) else .01,1e-4))
 dz[ids]=np.clip((rr-np.median(rr[ni],axis=0))/np.maximum(rr[ni].std(axis=0),floor),-20,20)
xx=np.vstack([embedding(Z[train]),embedding(dz)])
yy=np.r_[y[train],D.gas.to_numpy(int)]
m=LogisticRegression(C=.1,class_weight="balanced",max_iter=2000).fit(xx,yy,sample_weight=np.r_[np.ones(len(train)),np.full(len(D),.5)])
score5=m.predict_proba(embedding(Z[q]))[:,1]
# Direction checks and empirical midrank gates use known normal references only.
load_gate=np.zeros(len(U),bool);pressure_gate=np.zeros(len(U),bool);fast_gate=np.zeros(len(U),bool);meter_gate=np.zeros(len(U),bool)
for mode_name in T.run_id.unique():
 ni=normal[T.run_id.iloc[normal].eq(mode_name).to_numpy()]
 ids=np.flatnonzero(U.reference_mode.eq(mode_name))
 load_gate[ids]=(bp[q[ids]]>np.median(bp[ni]))&(rpm[q[ids]]<np.median(rpm[ni]))
 pressure_gate[ids]=(p[q[ids]]-sw[q[ids]])<np.median(p[ni]-sw[ni])
 for v,dest in [(speed,fast_gate),(meter,meter_gate)]:
  values=np.sort(np.round(v[ni],4));query=np.round(v[q[ids]],4)
  rank=(np.searchsorted(values,query,side="left")+np.searchsorted(values,query,side="right"))/(2*len(values))
  dest[ids]=rank>.95
arms={"relation_ET":score1>=cut1,"joint_LR":(score2>=cut2)&pressure_gate,
 "load_LR":(score3>=cut3)&load_gate,"persistent_ET":score4>=cut4,
 "transfer_LR":score5>=cut5,"local_flow_RF":(score6>=cut6)&fast_gate,
 "abs_flow_LR":(score7>=cut7)&meter_gate}
for name,v in arms.items():U[name]=v.astype(int)
for k,score in enumerate([score1,score2,score3,score4,score5,score6,score7],1):U["score_"+str(k)]=score
U["gas_candidate_raw"]=np.any(np.column_stack(list(arms.values())),axis=1).astype(int)
U["eligible_unlabeled"]=~U.known_label&~U.sensor_conflict&U.history_ready
# Require common history coverage for the whole7-head combination; don't silently impute startup into normal.
mask=U.eligible_unlabeled.to_numpy()
U["gas_candidate"]=np.where(mask,U.gas_candidate_raw,np.nan)
U.to_csv(OUT/"cn7_179_unlabeled_predictions.csv",index=False,encoding="utf-8-sig")
U[mask&U.gas_candidate_raw.eq(1)].to_csv(OUT/"cn7_179_unlabeled_gas_candidates.csv",index=False,encoding="utf-8-sig")
eligible_n=int(mask.sum());candidate_n=int(U.loc[mask,"gas_candidate_raw"].sum())
summary={"official_rows":len(official),"official_unique_shots":len(U),"known_labeled_shots_excluded":int(U.known_label.sum()),
 "unknown_shots":int((~U.known_label).sum()),"unknown_sensor_conflict":int((~U.known_label&U.sensor_conflict).sum()),
 "unknown_history_unavailable":int((~U.known_label&~U.history_ready).sum()),
 "eligible_unknown_shots":eligible_n,"gas_candidate_shots":candidate_n,"candidate_percent_of_eligible":100*candidate_n/eligible_n,
 "candidate_percent_of_all_unknown":100*candidate_n/int((~U.known_label).sum()),
 "mode_support_cut":support_cut,
 "supported_eligible":int((mask&U.in_reference_support).sum()),
 "supported_candidates":int((mask&U.in_reference_support&U.gas_candidate_raw.eq(1)).sum()),
 "model":"full605 fit same7 heads; mode matching to nearest labeled normal in3 control variables; frozen median fold cuts",
 "history":"past5 query sensor history from rawS14, reset30min/product switch; original sparse label history retained for training",
 "caveat":"No quality ground truth, not precision/recall/F1. Full-fit predictions are not the original0.714 OOF ensemble."}
summary["supported_candidate_percent"]=100*summary["supported_candidates"]/summary["supported_eligible"] if summary["supported_eligible"] else None
byrun=U[mask].groupby("history_run").agg(first=("TimeStamp","min"),last=("TimeStamp","max"),shots=("TimeStamp","size"),candidates=("gas_candidate_raw","sum"),supported=("in_reference_support","sum"))
byrun["candidate_percent"]=100*byrun.candidates/byrun.shots
byrun.to_csv(OUT/"cn7_179_unlabeled_by_run.csv",encoding="utf-8-sig")
modelcounts=pd.DataFrame([{"model":name,"candidate_shots":int(U.loc[mask,name].sum()),"percent":100*U.loc[mask,name].sum()/eligible_n} for name in arms])
modelcounts.to_csv(OUT/"cn7_179_unlabeled_by_model.csv",index=False)
row=official[["source_row_id","TimeStamp","side"]].merge(U[["TimeStamp","known_label","eligible_unlabeled","gas_candidate"]],on="TimeStamp",validate="many_to_one")
summary["eligible_rows"]=int(row.eligible_unlabeled.sum())
summary["candidate_rows"]=int(row.gas_candidate.fillna(0).sum())
summary["candidate_row_percent"]=100*summary["candidate_rows"]/summary["eligible_rows"]
(OUT/"cn7_179_unlabeled_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
(OUT/"cn7_179_frozen_thresholds.json").write_text(json.dumps(cuts,ensure_ascii=False,indent=2),encoding="utf-8")
row.to_csv(OUT/"cn7_179_unlabeled_row_mapping.csv",index=False,encoding="utf-8-sig")
assert not U.loc[mask,"TimeStamp"].isin(knownkeys).any()
assert all(np.isfinite(s).all() for s in [score1,score2,score3,score4,score5,score6,score7])
print(json.dumps(summary,ensure_ascii=False,indent=2))
print("\nMODELS\n",modelcounts.to_string(index=False))
print("\nRUNS TOP\n",byrun.sort_values("candidates",ascending=False).head(10).to_string())
lines=["# CN7 비라벨 가스 후보 경보 비율","",
 "최종7개 지도 모델을 전체 공식605샷으로 새 학습하고 적용했다. 원래F1 .714의 구간별 모델을 그대로 적용한 것이 아니다.",
 "기준은 기존 중첩 검증 각20블록의 임계값 중앙값으로 고정했다. 비라벨 결과를 보고 기준을 조정하지 않았다.",
 "공식 비라벨35239행/S14만 사용한다. 정규화된 공식 파일은Timestamp 선별에만 사용하고 실제 모델에는 연결한원시 센서를 넣었다.",
 "같은Timestamp의LH/RH를 한샷으로 처리한다. 이미 라벨이 있는606샷은 후보 비율에서 제외한다.",
 "예전Run ID를 새Run에 직접 할당할 수 없어 배럴5/호퍼/금형4가 가장 비슷한 학습 정상의운전모드를 선택했다.",
 "새Run의정답을 정상이라고 가정해 정규화하지 않았다. 참조정상은 라벨592샷만 사용한다.",
 "쿼리는원시비라벨 과거5샷이력을사용하므로 기존희소라벨 이력과밀도가달라진다. 초기3샷 미만/센서충돌은보류한다.",
 "정답이 없으므로 이하비율은가스경보비율이며 정확도/실제불량률이 아니다.","",
 "## 요약",json.dumps(summary,ensure_ascii=False,indent=2),"",
 "## 모델별(중복있음)",modelcounts.to_string(index=False),"",
 "## 후보가많은Run",byrun.sort_values("candidates",ascending=False).head(15).to_string(),"",
 "참조범위밖운전조건에서도 모델은큰점수를 낼수 있다. 지원범위내비율을따로 보고했지만 지원범위내라는것이정상/불량을보증하지는않는다."]
(ROOT/"reports/cn7_179_unlabeled_gas_application.md").write_text("\n".join(lines),encoding="utf-8")


{
  "official_rows": 35239,
  "official_unique_shots": 28744,
  "known_labeled_shots_excluded": 606,
  "unknown_shots": 28138,
  "unknown_sensor_conflict": 28,
  "unknown_history_unavailable": 370,
  "eligible_unknown_shots": 27768,
  "gas_candidate_shots": 7765,
  "candidate_percent_of_eligible": 27.963843272832037,
  "candidate_percent_of_all_unknown": 27.596133342810436,
  "mode_support_cut": 0.5017312249266509,
  "supported_eligible": 2582,
  "supported_candidates": 1491,
  "model": "full605 fit same7 heads; mode matching to nearest labeled normal in3 control variables; frozen median fold cuts",
  "history": "past5 query sensor history from rawS14, reset30min/product switch; original sparse label history retained for training",
  "caveat": "No quality ground truth, not precision/recall/F1. Full-fit predictions are not the original0.714 OOF ensemble.",
  "supported_candidate_percent": 57.745933384972886,
  "eligible_rows": 33464,
  "candidate_rows": 11736,
  "candidate_row_percent":

3677